In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupKFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

ROOT = Path.cwd().parents[1]
DATA_DIR = ROOT / "data"

print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)

df = pd.read_csv(DATA_DIR / "duolingo_flagship_v5.csv")
split_users = pd.read_csv(DATA_DIR / "split_users.csv")

cv_users = split_users.loc[split_users["split"] == "cv","user_id"]
cv_df = df[df["user_id"].isin(cv_users)].copy()

features = [
    "lag_days",
    "history_seen",
    "history_correct",
    "history_accuracy",
    "lag_days_log"
]

target = "p_recall"

X = cv_df[features]
y = cv_df[target]
groups = cv_df["user_id"]

gkf = GroupKFold(n_splits=5)

print("CV rows:", len(cv_df))
print("CV users:", cv_df["user_id"].nunique())
print("X shape:", X.shape)

ROOT: C:\Users\kutay\Desktop\ml-learning
DATA_DIR: C:\Users\kutay\Desktop\ml-learning\data
CV rows: 14438
CV users: 2125
X shape: (14438, 5)


In [2]:
fold_rmses = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups),start=1):
    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    model = RandomForestRegressor(
        n_estimators=200,
        max_features=2,
        bootstrap=True,
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, pred))
    fold_rmses.append(rmse)
    print(f"Fold {fold}: {rmse:.6f}")

print()
print(f"Mean RMSE: {np.mean(fold_rmses):.6f}")
print(f"Std RMSE:  {np.std(fold_rmses):.6f}")

Fold 1: 0.307209
Fold 2: 0.303938
Fold 3: 0.290302
Fold 4: 0.289471
Fold 5: 0.312238

Mean RMSE: 0.300632
Std RMSE:  0.009167


In [3]:
max_features_values = [1, 2, 3, 4, 5]

results = []

for max_features in max_features_values:
    fold_rmses = []

    for train_idx, val_idx in gkf.split(X, y, groups):
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_val = X.iloc[val_idx]
        y_val = y.iloc[val_idx]

        model = RandomForestRegressor(
            n_estimators=200,
            max_features=max_features,
            bootstrap=True,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)
        pred = model.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, pred))
        fold_rmses.append(rmse)

    results.append({
        "max_features": max_features,
        "mean_rmse": np.mean(fold_rmses),
        "std_rmse": np.std(fold_rmses)
    })

pd.DataFrame(results)

,max_features,mean_rmse,std_rmse
0,1,0.303399,0.008637
1,2,0.300632,0.009167
2,3,0.298876,0.009244
3,4,0.298666,0.009107
4,5,0.299087,0.009118


In [4]:
tree_counts = [10, 20, 50, 100, 200, 400]

results_trees = []

for B in tree_counts:
    fold_rmses = []

    for train_idx, val_idx in gkf.split(X, y, groups):
        X_train = X.iloc[train_idx]
        y_train = y.iloc[train_idx]

        X_val = X.iloc[val_idx]
        y_val = y.iloc[val_idx]

        model = RandomForestRegressor(
            n_estimators=B,
            max_features=4,
            bootstrap=True,
            random_state=42,
            n_jobs=-1
        )

        model.fit(X_train, y_train)
        pred = model.predict(X_val)

        rmse = np.sqrt(mean_squared_error(y_val, pred))
        fold_rmses.append(rmse)

    results_trees.append({
        "n_estimators": B,
        "mean_rmse": np.mean(fold_rmses),
        "std_rmse": np.std(fold_rmses)
    })

pd.DataFrame(results_trees)

,n_estimators,mean_rmse,std_rmse
0,10,0.307334,0.008597
1,20,0.303255,0.008694
2,50,0.300331,0.009037
3,100,0.299055,0.009050
4,200,0.298666,0.009107
5,400,0.298430,0.009287


## Part 14 — Random Forest

Random Forest extends bagging by adding feature randomness.

At each node:
- randomly select a subset of features
- search for the best split only within that subset

This reduces correlation between trees and can make averaging more effective.

### Strength-diversity tradeoff

Smaller `max_features`:
- diversity ↑
- tree correlation ↓
- individual tree strength may ↓

Larger `max_features`:
- stronger individual trees
- tree correlation ↑

### Flagship results

Using 200 trees:

- max_features=1 → RMSE ≈ 0.3034
- max_features=2 → RMSE ≈ 0.3006
- max_features=3 → RMSE ≈ 0.2989
- max_features=4 → RMSE ≈ 0.2987
- max_features=5 → RMSE ≈ 0.2991

With `max_features=4`:

- 10 trees → RMSE ≈ 0.3073
- 100 trees → RMSE ≈ 0.2991
- 200 trees → RMSE ≈ 0.2987
- 400 trees → RMSE ≈ 0.2984

### Interpretation

Feature randomness gave a small improvement over plain bagging, but gains were limited.

`max_features=1` weakened individual trees too much.
`max_features=5` removed feature randomness and approached ordinary bagging.
`max_features=4` gave the lowest observed CV mean RMSE, suggesting a reasonable strength-diversity balance.

Increasing the number of trees showed diminishing returns and approached a performance plateau.